# HS4 Code Prediction
### Product descriptions → ranked tariff-heading candidates
**Author: Ayesha Ansari**

This portfolio project compares word and character TF-IDF features with linear support vector machines. It predicts a **4-digit HS heading**, rather than a complete country-specific tariff code.

**How to run:** In Google Colab, select **Runtime → Run all**. The notebook downloads the source data, trains three configurations, evaluates the selected model, and writes model and CSV artifacts. Training can take several minutes on CPU. Outputs are intentionally empty in this supplied notebook; run it to produce your own results.

**Decisions fixed in the previous development run:** word + character calibrated SVM; provisional review threshold **0.50**. Re-running this notebook does not automatically select another model or tune on test results.

**Scope:** Educational decision support with human review. Source labels are extracted from dataset answers and have not been independently verified against customs rulings.

## 1. Setup
Run this setup cell first. It pins scikit-learn to the version used for the current deployment. If Colab asks you to restart after installation, restart the session and run from the top. Training dependency versions and the resolved dataset revision are recorded with the exported model.

In [1]:
%pip install -q scikit-learn==1.6.1 datasets joblib

In [2]:
import ast
import json
import os
import platform
import re
from pathlib import Path

import datasets
import joblib
import numpy as np
import pandas as pd
import sklearn
from datasets import load_dataset
from huggingface_hub import HfApi
from IPython.display import display
from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.svm import LinearSVC

SEED = 42
MIN_SAMPLES = 10
DATASET_ID = "Dayanand314Krishna/cross_rulings_hts_dataset_for_tariffs"
# To reproduce a particular run, replace None with the revision saved in model_info.json.
DATASET_REVISION = None
OUTPUT_DIR = Path("hs4_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
print("scikit-learn:", sklearn.__version__)
assert sklearn.__version__ == "1.6.1", "Restart the session after installation."

scikit-learn: 1.6.1


## 2. Load the dataset
Only the source **training split** is used to construct this project's own splits. Resolve a commit revision before loading so this run can be reproduced later.

Dataset: [CROSS Rulings HTS dataset](https://huggingface.co/datasets/Dayanand314Krishna/cross_rulings_hts_dataset_for_tariffs). Attribution: CROSS Rulings HTS Dataset for Tariff Classification by Flexify.AI Inc. (https://www.flexify.ai). Check the source dataset terms before redistribution.

In [3]:
resolved_revision = DATASET_REVISION or HfApi().dataset_info(DATASET_ID).sha
raw_df = load_dataset(DATASET_ID, revision=resolved_revision, split="train").to_pandas()
print("Dataset revision:", resolved_revision)
print("Original shape:", raw_df.shape)

README.md:   0%|          | 0.00/3.98k [00:00<?, ?B/s]

train.jsonl: reconstructing file:   0%|          |  0.00B / 23.7MB            

train.jsonl: downloading bytes:           |  0.00B            

validation.jsonl:   0%|          | 0.00/260k [00:00<?, ?B/s]

test.jsonl:   0%|          | 0.00/257k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Dataset revision: 0a66ad345d3fc9828f362836d86b31e20798513c
Original shape: (18254, 1)


## 3. Extract product descriptions and labels
Use the first user message as the input and the explicit `HTS US Code ->` answer field as the label source. Reasoning text is never used as a feature.

Accept dotted codes only. Several complete codes may be retained if they share one HS4 heading. Exclude missing or multiple different headings. This conservative rule also excludes some usable cases with additional Chapter 99 tariff provisions. It cannot resolve contradictions within source answers.

In [4]:
def extract_text_and_code(message):
    if isinstance(message, (list, np.ndarray)):
        data = list(message)
    else:
        try:
            data = ast.literal_eval(str(message))
        except (ValueError, SyntaxError):
            return "", None
    if not isinstance(data, list):
        return "", None

    user_text, assistant_text = "", ""
    for item in data:
        if not isinstance(item, dict):
            continue
        content = item.get("content", "")
        if not isinstance(content, str):
            continue
        if item.get("role") == "user" and not user_text:
            user_text = content
        elif item.get("role") == "assistant" and not assistant_text:
            assistant_text = content

    user_text = re.sub(
        r"^(what\s+is\s+the\s+hts\s+(us\s+)?code\s+for\s+|"
        r"what\s+is\s+the\s+hs\s+code\s+for\s+)",
        "", user_text.strip(), flags=re.IGNORECASE
    ).rstrip("?").lower().strip()

    answer_match = re.search(
        r"HTS\s+US\s+Code\s*->\s*(.*?)(?=\s*Reasoning\s*->|$)",
        assistant_text, flags=re.IGNORECASE | re.DOTALL
    )
    if not answer_match:
        return user_text, None
    candidates = set(re.findall(
        r"(?<![\w.])(\d{4})\.\d{2}(?!\d)", answer_match.group(1)
    ))
    if len(candidates) != 1:
        return user_text, None
    return user_text, next(iter(candidates))

parsed = raw_df["messages"].apply(extract_text_and_code)
parsed_df = pd.DataFrame(parsed.tolist(), columns=["clean_text", "hs4"])
print("Records with an extracted label:", parsed_df["hs4"].notna().sum())

Records with an extracted label: 16123


## 4. Audit extraction and clean the data
Inspect sampled accepted and rejected source answers. This checks extraction behavior, not the legal correctness of labels. Then remove invalid descriptions, conflicting exact descriptions, exact duplicates and classes with fewer than ten examples.

These rules define a **filtered, supported-class benchmark** before splitting. Conflict removal and class eligibility use labels across this source pool; the evaluation is conditional on that curation and does not measure performance across all incoming products.

In [5]:
audit_df = parsed_df.copy()
audit_df["raw_messages"] = raw_df["messages"]
for accepted, title in [(True, "Accepted"), (False, "Rejected")]:
    subset = audit_df[audit_df["hs4"].notna() == accepted]
    print(f"\n{title} examples")
    display(subset.sample(n=min(5, len(subset)), random_state=SEED))


Accepted examples


,clean_text,hs4,raw_messages
4846,mens knit pants constructed from 100% polyeste...,6103,"[{'role': 'user', 'content': 'What is the HTS ..."
1515,a boys jacket made of 100% knit polyester fabr...,6101,"[{'role': 'user', 'content': 'What is the HTS ..."
13057,setting-up sample discs designed for routine d...,9027,"[{'role': 'user', 'content': 'What is the HTS ..."
6781,pacing leads used in pacemakers,9021,"[{'role': 'user', 'content': 'What is the HTS ..."
6024,a high-speed pressure water descaler unit used...,8424,"[{'role': 'user', 'content': 'What is the HTS ..."



Rejected examples


,clean_text,hs4,raw_messages
2570,"benzyl alcohol, benzyl acetate, and benzyl ben...",None,"[{'role': 'user', 'content': 'What is the HTS ..."
17290,barbecue grids and grills made of steel and po...,None,"[{'role': 'user', 'content': 'What is the HTS ..."
14639,plastic and stainless steel containers used fo...,None,"[{'role': 'user', 'content': 'What is the HTS ..."
14241,a women's knit coat made of 100% wool and a wo...,None,"[{'role': 'user', 'content': 'What is the HTS ..."
17733,unfinished tee fittings made of carbon steel,None,"[{'role': 'user', 'content': 'What is the HTS ..."


In [6]:
valid_df = parsed_df.dropna(subset=["clean_text", "hs4"]).copy()
valid_df = valid_df[
    valid_df["hs4"].str.fullmatch(r"\d{4}", na=False)
    & (valid_df["clean_text"].str.len() > 10)
].copy()
conflicting = valid_df.groupby("clean_text")["hs4"].nunique()
conflicting_texts = conflicting[conflicting > 1].index
unambiguous_df = valid_df[~valid_df["clean_text"].isin(conflicting_texts)].copy()
pre_filter_df = unambiguous_df.drop_duplicates(["clean_text", "hs4"]).reset_index(drop=True)
counts = pre_filter_df["hs4"].value_counts()
valid_classes = counts[counts >= MIN_SAMPLES].index
df = pre_filter_df[pre_filter_df["hs4"].isin(valid_classes)].copy()

funnel = pd.DataFrame({
    "Stage": ["Original source records", "Valid codes and descriptions",
              "After conflicting descriptions removed", "After exact duplicate removal",
              "After rare-class filter"],
    "Records": [len(raw_df), len(valid_df), len(unambiguous_df), len(pre_filter_df), len(df)]
})
display(funnel)
source_codes = set(parsed_df["hs4"].dropna())
supported_codes = set(df["hs4"])
record_retention = len(df) / len(pre_filter_df)
distinct_code_coverage = len(supported_codes & source_codes) / len(source_codes)
print(f"Supported HS4 headings: {len(supported_codes)} / {len(source_codes)} parsed headings")
print(f"Record retention after rare-class filtering: {record_retention:.2%}")
print(f"Distinct parsed-code coverage: {distinct_code_coverage:.2%}")
funnel.to_csv(OUTPUT_DIR / "coverage_funnel.csv", index=False)

# Flag code-like strings in inputs for inspection; do not automatically delete numbers.
# Measurements and model numbers may legitimately contain four-digit numbers.
code_like = df["clean_text"].str.contains(r"\b\d{4}\.\d{2}\b", regex=True)
print("Descriptions containing dotted code-like strings:", int(code_like.sum()))
if code_like.any():
    display(df.loc[code_like, ["clean_text", "hs4"]].head(10))
print("Review flagged inputs before publishing performance claims.")

,Stage,Records
0,Original source records,18254
1,Valid codes and descriptions,16103
2,After conflicting descriptions removed,15546
3,After exact duplicate removal,13996
4,After rare-class filter,13240


Supported HS4 headings: 236 / 495 parsed headings
Record retention after rare-class filtering: 94.60%
Distinct parsed-code coverage: 47.68%
Descriptions containing dotted code-like strings: 2


,clean_text,hs4
10126,a framed glass mirror designed for general hou...,7009
11188,a framed glass mirror designed for a child's r...,7009


Review flagged inputs before publishing performance claims.


## 5. Create training, validation and test splits
Use approximately **60% / 20% / 20%**, stratified by heading. Fit models on training only; use validation for comparisons and threshold analysis. Test results are reported after decisions are fixed.

Exact-text overlap is checked. Random splits may still share related products or rulings. These data have been inspected during development, so this is not a newly collected external benchmark.

In [7]:
X_development, X_test, y_development, y_test = train_test_split(
    df["clean_text"], df["hs4"], test_size=0.20,
    random_state=SEED, stratify=df["hs4"]
)
X_train, X_val, y_train, y_val = train_test_split(
    X_development, y_development, test_size=0.25,
    random_state=SEED, stratify=y_development
)
assert set(X_train).isdisjoint(set(X_val))
assert set(X_train).isdisjoint(set(X_test))
assert set(X_val).isdisjoint(set(X_test))
assert y_train.value_counts().min() >= 3
print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))
print("Test samples:", len(X_test))
print("Supported classes:", y_train.nunique())
print("Exact-text overlap checks passed.")
split_membership = pd.concat([
    pd.DataFrame({"record_index": X.index, "split": name})
    for name, X in [("train", X_train), ("validation", X_val), ("test", X_test)]
], ignore_index=True)
split_membership.to_csv(OUTPUT_DIR / "split_membership.csv", index=False)

Training samples: 7944
Validation samples: 2648
Test samples: 2648
Supported classes: 236
Exact-text overlap checks passed.


## 6. Train the calibrated word + character model
Word TF-IDF uses 1–2 word sequences; character TF-IDF uses 3–5 character sequences. Both are fitted **inside** each calibration fold. Calibration therefore does not learn preprocessing from its held-out fold. The outer validation and test sets are excluded from fitting.

In [8]:
base_pipeline = Pipeline([
    ("features", FeatureUnion([
        ("word", TfidfVectorizer(analyzer="word", ngram_range=(1, 2),
            min_df=2, max_features=30000, sublinear_tf=True)),
        ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5),
            min_df=2, max_features=30000, sublinear_tf=True))
    ])),
    ("svm", LinearSVC(C=1.0, class_weight="balanced", max_iter=5000, random_state=SEED))
])
model = CalibratedClassifierCV(estimator=base_pipeline, cv=3, method="sigmoid", n_jobs=-1)
print("Training calibrated word + character SVM...")
model.fit(X_train, y_train)
print("Training complete.")

Training calibrated word + character SVM...
Training complete.


## 7. Compare models on validation data
Top-1 measures the first candidate; Top-3 checks whether the target appears among three candidates. Macro F1 weights headings equally and helps expose weak performance on less frequent classes. Model comparisons are exploratory; small differences are not proof of superiority.

In [9]:
def rank_predictions(fitted_model, texts, calibrated=True):
    scores = (fitted_model.predict_proba(texts) if calibrated
              else fitted_model.decision_function(texts))
    classes = np.asarray(fitted_model.classes_).astype(str)
    ranked = np.argsort(scores, axis=1)[:, ::-1]
    return classes[ranked[:, 0]], classes[ranked[:, :3]]

def evaluate_ranked(name, actual, top1, top3, classes):
    actual = np.asarray(actual).astype(str)
    return {"Model": name, "Top-1 Accuracy": accuracy_score(actual, top1),
            "Top-3 Accuracy": np.mean([a in candidates for a, candidates in zip(actual, top3)]),
            "Macro F1": f1_score(actual, top1, labels=classes, average="macro", zero_division=0)}

print("Training word-only comparison model...")
word_only_pipeline = clone(base_pipeline)
word_only_pipeline.set_params(features=TfidfVectorizer(
    analyzer="word", ngram_range=(1, 2), min_df=2, max_features=30000, sublinear_tf=True))
word_only_model = CalibratedClassifierCV(
    estimator=word_only_pipeline, cv=3, method="sigmoid", n_jobs=-1)
word_only_model.fit(X_train, y_train)

print("Training uncalibrated comparison model...")
raw_pipeline = clone(base_pipeline)
raw_pipeline.fit(X_train, y_train)
comparison_rows = []
for name, estimator, calibrated in [
    ("Word + Character, Calibrated SVM", model, True),
    ("Word-only, Calibrated SVM", word_only_model, True),
    ("Word + Character, Uncalibrated SVM", raw_pipeline, False)
]:
    top1, top3 = rank_predictions(estimator, X_val, calibrated)
    comparison_rows.append(evaluate_ranked(name, y_val, top1, top3, estimator.classes_))
comparison = pd.DataFrame(comparison_rows)
display(comparison)
comparison.to_csv(OUTPUT_DIR / "validation_model_comparison.csv", index=False)

Training word-only comparison model...
Training uncalibrated comparison model...


,Model,Top-1 Accuracy,Top-3 Accuracy,Macro F1
0,"Word + Character, Calibrated SVM",0.688822,0.813822,0.566774
1,"Word-only, Calibrated SVM",0.645015,0.774547,0.498884
2,"Word + Character, Uncalibrated SVM",0.694109,0.807779,0.586056


## 8. Examine validation confidence and thresholds
Calibration is a training method, not a guarantee of reliable probabilities. This five-bin top-label ECE summarizes the gap between average maximum probability and observed Top-1 accuracy. Its value depends on the bins.

Threshold coverage below means the fraction of validation cases scoring at least the threshold. It is different from dataset record retention. The analysis contains supported labels only and does **not** validate detection of missing classes.

In [10]:
# The highest predicted probability is the Top-1 confidence
val_probabilities = model.predict_proba(X_val)
val_classes = np.asarray(model.classes_).astype(str)
val_predictions = val_classes[val_probabilities.argmax(axis=1)]
confidence = val_probabilities.max(axis=1)
correct = val_predictions == np.asarray(y_val).astype(str)

reliability_df = pd.DataFrame({
    "confidence": confidence,
    "correct": correct
})

# Group confidence into five ranges
bins = [0, 0.2, 0.4, 0.6, 0.8, 1.000001]
labels = ["0-20%", "20-40%", "40-60%", "60-80%", "80-100%"]

reliability_df["confidence_bin"] = pd.cut(
    reliability_df["confidence"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

reliability_table = reliability_df.groupby(
    "confidence_bin",
    observed=False
).agg(
    Samples=("correct", "size"),
    Mean_Confidence=("confidence", "mean"),
    Actual_Accuracy=("correct", "mean")
).reset_index()

# Exclude empty bins from the calibration-error calculation
nonempty = reliability_table[
    reliability_table["Samples"] > 0
].copy()

ece = np.sum(
    nonempty["Samples"] / len(reliability_df)
    * (
        nonempty["Mean_Confidence"]
        - nonempty["Actual_Accuracy"]
    ).abs()
)

# Convert table values to percentages
reliability_table["Mean_Confidence"] *= 100
reliability_table["Actual_Accuracy"] *= 100

print("CONFIDENCE RELIABILITY")
print("=" * 40)
print(reliability_table.to_string(index=False))
print(f"\nExpected Calibration Error (ECE): {ece:.4f}")

# Confidence threshold analysis
print("\nCONFIDENCE THRESHOLDS")

for threshold in [0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90]:
    selected = confidence >= threshold

    if selected.any():
        accuracy = correct[selected].mean()

        print(
            f"Confidence >= {threshold:.0%}: "
            f"{selected.sum()}/{len(correct)} cases "
            f"({selected.mean():.2%} coverage), "
            f"Top-1 accuracy = {accuracy:.2%}"
        )
    else:
        print(f"Confidence >= {threshold:.0%}: no cases")

reliability_table.to_csv(
    OUTPUT_DIR / "validation_confidence_reliability.csv",
    index=False
)

print("\nSaved validation_confidence_reliability.csv")
validation_ece = float(ece)

CONFIDENCE RELIABILITY
confidence_bin  Samples  Mean_Confidence  Actual_Accuracy
         0-20%      419        12.687280        23.866348
        20-40%      568        29.923378        49.647887
        40-60%      613        49.970527        73.409462
        60-80%     1014        71.343830        94.477318
       80-100%       34        80.709114       100.000000

Expected Calibration Error (ECE): 0.2053

CONFIDENCE THRESHOLDS
Confidence >= 30%: 1929/2648 cases (72.85% coverage), Top-1 accuracy = 82.27%
Confidence >= 40%: 1661/2648 cases (62.73% coverage), Top-1 accuracy = 86.82%
Confidence >= 50%: 1345/2648 cases (50.79% coverage), Top-1 accuracy = 90.56%
Confidence >= 60%: 1048/2648 cases (39.58% coverage), Top-1 accuracy = 94.66%
Confidence >= 70%: 654/2648 cases (24.70% coverage), Top-1 accuracy = 97.71%
Confidence >= 80%: 34/2648 cases (1.28% coverage), Top-1 accuracy = 100.00%
Confidence >= 90%: no cases

Saved validation_confidence_reliability.csv


## 9. Record the fixed model and review threshold
The previous validation run favored the uncalibrated model on Top-1 and Macro F1 and the calibrated model on Top-3. The calibrated word + character model was retained for candidate ranking, with **0.50** as a provisional review threshold. These decisions are fixed for this notebook; do not change them after reviewing test scores.

Descriptions below 0.50 require an uncertainty warning. Scores above 0.50 still require human verification; they do not prove the true class is supported.

In [11]:
SELECTED_MODEL = "Word + Character, Calibrated SVM"
REVIEW_THRESHOLD = 0.50
print("Selected model:", SELECTED_MODEL)
print("Review threshold:", REVIEW_THRESHOLD)

Selected model: Word + Character, Calibrated SVM
Review threshold: 0.5


## 10. Final test evaluation and error export
Evaluate the fixed model on all supported-class test records and against a majority-class baseline learned from training labels. Save both all predictions and incorrect Top-1 predictions. Report threshold accuracy together with coverage; it is not overall accuracy.

In [12]:
test_probs = model.predict_proba(X_test)
classes = np.asarray(model.classes_).astype(str)
y_true = np.asarray(y_test).astype(str)
ranked = np.argsort(test_probs, axis=1)[:, ::-1]
predictions = classes[ranked[:, 0]]
top3_labels = classes[ranked[:, :3]]
metrics = evaluate_ranked(SELECTED_MODEL, y_true, predictions, top3_labels, classes)
top1_accuracy = metrics["Top-1 Accuracy"]
top3_accuracy = metrics["Top-3 Accuracy"]
macro_f1 = metrics["Macro F1"]
majority_code = str(y_train.value_counts().idxmax())
baseline_accuracy = accuracy_score(y_true, np.full(len(y_true), majority_code))
print(f"Top-1 accuracy: {top1_accuracy:.2%}")
print(f"Top-3 accuracy: {top3_accuracy:.2%}")
print(f"Macro F1: {macro_f1:.4f}")
print(f"Majority baseline: {baseline_accuracy:.2%} (heading {majority_code})")
print("Test samples:", len(y_true))
print("Correct Top-1:", int((predictions == y_true).sum()))
print("Incorrect Top-1:", int((predictions != y_true).sum()))

report = pd.DataFrame(classification_report(
    y_true, predictions, labels=classes, output_dict=True, zero_division=0)).T
report.to_csv(OUTPUT_DIR / "classification_report.csv")
display(report.tail(3))

results_df = pd.DataFrame({
    "text": X_test.to_numpy(), "actual_hs4": y_true, "predicted_hs4": predictions,
    "top1_correct": predictions == y_true,
    "top3_predictions": [" | ".join(row) for row in top3_labels],
    "confidence": np.round(test_probs.max(axis=1) * 100, 2)
})
results_df.to_csv(OUTPUT_DIR / "test_predictions.csv", index=False)
results_df.loc[~results_df["top1_correct"]].to_csv(OUTPUT_DIR / "wrong_predictions.csv", index=False)

above_threshold = test_probs.max(axis=1) >= REVIEW_THRESHOLD
selected_accuracy = (float((predictions[above_threshold] == y_true[above_threshold]).mean())
                     if above_threshold.any() else None)
print(f"\nFixed threshold: {REVIEW_THRESHOLD:.2f}")
print(f"Cases above threshold: {above_threshold.sum()}/{len(y_true)}")
print(f"Threshold coverage: {above_threshold.mean():.2%}")
if selected_accuracy is not None:
    print(f"Top-1 accuracy above threshold: {selected_accuracy:.2%}")
summary = {
    "Top-1 Accuracy": top1_accuracy, "Top-3 Accuracy": top3_accuracy,
    "Macro F1": macro_f1, "Majority-Class Baseline": baseline_accuracy,
    "Test Samples": len(y_true), "Review Threshold": REVIEW_THRESHOLD,
    "Threshold Coverage": float(above_threshold.mean()),
    "Accuracy Above Threshold": selected_accuracy
}
pd.DataFrame(summary.items(), columns=["Metric", "Value"]).to_csv(
    OUTPUT_DIR / "model_evidence_summary.csv", index=False)

Top-1 accuracy: 67.37%
Top-3 accuracy: 79.53%
Macro F1: 0.5294
Majority baseline: 7.36% (heading 6110)
Test samples: 2648
Correct Top-1: 1784
Incorrect Top-1: 864


,precision,recall,f1-score,support
accuracy,0.673716,0.673716,0.673716,0.673716
macro avg,0.573222,0.527481,0.529445,2648.000000
weighted avg,0.664323,0.673716,0.658583,2648.000000



Fixed threshold: 0.50
Cases above threshold: 1334/2648
Threshold coverage: 50.38%
Top-1 accuracy above threshold: 91.00%


## 11. Diagnose train–test similarity
Fit the diagnostic TF-IDF representation on training data only, then find each test description's nearest training description. High cosine similarity is a warning to inspect related examples, not proof of leakage. Low similarity can indicate less familiar wording.

This is a post-evaluation diagnostic. No test cases are removed and no model decisions are changed based on this table. A stronger future benchmark would group related products or source rulings before splitting.

In [13]:
diagnostic_features = raw_pipeline.named_steps["features"]
X_train_features = diagnostic_features.transform(X_train)
X_test_features = diagnostic_features.transform(X_test)

# ============================================================
# NEAR-DUPLICATE SIMILARITY CHECK
# ============================================================

from sklearn.neighbors import NearestNeighbors

nearest_model = NearestNeighbors(
    n_neighbors=1,
    metric="cosine",
    algorithm="brute",
    n_jobs=-1
)

nearest_model.fit(X_train_features)

distances, indices = nearest_model.kneighbors(X_test_features)

similarity = np.clip(1 - distances.ravel(), 0, 1)

actual = np.asarray(y_test).astype(str)

combined_probs = model.predict_proba(X_test)
combined_classes = model.classes_.astype(str)

top1 = combined_classes[np.argmax(combined_probs, axis=1)]
top3 = combined_classes[
    np.argsort(combined_probs, axis=1)[:, -3:][:, ::-1]
]

similarity_df = pd.DataFrame({
    "test_text": X_test.to_numpy(),
    "actual_hs4": actual,
    "predicted_hs4": top1,
    "top1_correct": top1 == actual,
    "top3_correct": [
        label in candidates
        for label, candidates in zip(actual, top3)
    ],
    "nearest_train_text": X_train.iloc[indices.ravel()].to_numpy(),
    "nearest_similarity": similarity
})

bins = [0, 0.25, 0.50, 0.70, 0.85, 1.000001]
labels = ["0–0.25", "0.25–0.50", "0.50–0.70",
          "0.70–0.85", "0.85–1.00"]

similarity_df["similarity_band"] = pd.cut(
    similarity_df["nearest_similarity"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

report = similarity_df.groupby(
    "similarity_band",
    observed=False
).agg(
    samples=("top1_correct", "size"),
    mean_similarity=("nearest_similarity", "mean"),
    top1_accuracy=("top1_correct", "mean"),
    top3_accuracy=("top3_correct", "mean")
).reset_index()

for column in ["mean_similarity", "top1_accuracy", "top3_accuracy"]:
    report[column] *= 100

print("ACCURACY BY NEAREST-TRAINING SIMILARITY")
print(report.to_string(index=False))

print("\nHigh-similarity test cases (similarity >= 0.90):")

high = similarity_df["nearest_similarity"] >= 0.90
print("Cases:", int(high.sum()))

if high.any():
    print("Top-1 accuracy:",
          f"{similarity_df.loc[high, 'top1_correct'].mean():.2%}")
    print("Top-3 accuracy:",
          f"{similarity_df.loc[high, 'top3_correct'].mean():.2%}")

similarity_df.to_csv(OUTPUT_DIR / "near_duplicate_analysis.csv", index=False)

ACCURACY BY NEAREST-TRAINING SIMILARITY
similarity_band  samples  mean_similarity  top1_accuracy  top3_accuracy
         0–0.25      100        22.955699      32.000000      49.000000
      0.25–0.50     1457        37.809719      58.407687      73.644475
      0.50–0.70      608        58.742200      78.453947      88.815789
      0.70–0.85      276        77.153989      89.130435      92.753623
      0.85–1.00      207        92.010543      85.990338      90.821256

High-similarity test cases (similarity >= 0.90):
Cases: 129
Top-1 accuracy: 82.95%
Top-3 accuracy: 89.15%


## 12. Try a description
This is an illustrative example, not another performance benchmark. The model can only rank learned classes. A low score does not identify why a case is uncertain, and a high score does not establish that its true heading is supported.

In [14]:
sample = "Women's 100% woven cotton summer dress with floral embroidery"
sample_probs = model.predict_proba([sample.lower().strip()])[0]
print("Product:", sample)
print("Manual review warning:", bool(sample_probs.max() < REVIEW_THRESHOLD))
for rank, idx in enumerate(np.argsort(sample_probs)[::-1][:3], start=1):
    print(f"{rank}. HS4 {classes[idx]} — model score {sample_probs[idx]:.2%}")

Product: Women's 100% woven cotton summer dress with floral embroidery
Manual review warning: True
1. HS4 6110 — model score 9.28%
2. HS4 3926 — model score 7.83%
3. HS4 6108 — model score 4.34%


## 13. Save and verify model artifacts
Save the **same model that was evaluated**, without refitting on validation or test data. Reload it and compare predictions. Package versions, the resolved dataset revision and fixed threshold are recorded for reproducibility.

For deployment, use the same package versions. The app must load this new artifact and use threshold 0.50. The ZIP contains generated results and the model; the notebook itself is downloaded through Colab's File menu.

In [15]:
model_info = {
    "model_type": SELECTED_MODEL, "target": "HS4",
    "num_classes": len(classes), "classes": classes.tolist(),
    "dataset_id": DATASET_ID, "dataset_revision": resolved_revision,
    "training_samples": len(X_train), "validation_samples": len(X_val),
    "test_samples": len(X_test), "random_state": SEED,
    "minimum_class_samples": MIN_SAMPLES,
    "top1_accuracy": float(top1_accuracy), "top3_accuracy": float(top3_accuracy),
    "macro_f1": float(macro_f1), "majority_baseline": float(baseline_accuracy),
    "review_threshold": REVIEW_THRESHOLD, "threshold_selected_on": "validation",
    "threshold_test_coverage": float(above_threshold.mean()),
    "threshold_test_accuracy": selected_accuracy,
    "record_retention": record_retention,
    "distinct_parsed_code_coverage": distinct_code_coverage,
    "validation_top_label_ece_5_bins": validation_ece,
    "calibration": "sigmoid", "calibration_cv": 3,
    "tfidf_fitted_within_calibration_folds": True,
    "python_version": platform.python_version(),
    "sklearn_version": sklearn.__version__, "numpy_version": np.__version__,
    "pandas_version": pd.__version__, "joblib_version": joblib.__version__,
    "datasets_version": datasets.__version__
}
model_path = OUTPUT_DIR / "final_hs4_calibrated_svm.pkl"
joblib.dump(model, model_path, compress=3)
joblib.dump(model_info, OUTPUT_DIR / "model_info.pkl")
(OUTPUT_DIR / "model_info.json").write_text(json.dumps(model_info, indent=2))
reloaded_model = joblib.load(model_path)
np.testing.assert_allclose(
    model.predict_proba(X_val.iloc[:5]), reloaded_model.predict_proba(X_val.iloc[:5]),
    rtol=1e-10, atol=1e-12)
print("Save/reload prediction check passed.")
for path in sorted(OUTPUT_DIR.iterdir()):
    print(f"{path.name}: {path.stat().st_size / (1024 ** 2):.2f} MB")

import shutil
archive_path = shutil.make_archive("hs4_project_outputs", "zip", OUTPUT_DIR)
print("Output archive:", archive_path)

Save/reload prediction check passed.
classification_report.csv: 0.01 MB
coverage_funnel.csv: 0.00 MB
final_hs4_calibrated_svm.pkl: 66.72 MB
model_evidence_summary.csv: 0.00 MB
model_info.json: 0.00 MB
model_info.pkl: 0.00 MB
near_duplicate_analysis.csv: 0.56 MB
split_membership.csv: 0.15 MB
test_predictions.csv: 0.33 MB
validation_confidence_reliability.csv: 0.00 MB
validation_model_comparison.csv: 0.00 MB
wrong_predictions.csv: 0.10 MB
Output archive: /content/hs4_project_outputs.zip


## Interpretation and limitations
- Report overall Top-1, Top-3 and Macro F1 together. Accuracy above the review threshold applies only to the selected subset and must include its coverage.
- Source labels may be noisy or contradictory. The parser verifies format and explicit answer consistency, not customs correctness.
- The benchmark excludes ambiguous descriptions and rare classes, and uses random splits after dataset curation. It does not establish performance across all products or unseen ruling families.
- Exact duplicates are removed, but related or near-duplicate descriptions can remain across splits.
- Probability estimates can remain substantially underconfident after calibration. The threshold is a review aid, not a validated unsupported-class detector.
- Dataset revision and package versions are recorded. Input audits and a future external/grouped evaluation remain important before stronger claims.

**Download in Colab:** use the Files sidebar to download `hs4_project_outputs.zip`. Save the executed notebook with **File → Download → Download .ipynb**. Review the rendered notebook before uploading it to GitHub.